## Web Search Agent

In [1]:
!uv add  langchain-community langchain-core langchain_huggingface requests  python-dotenv duckduckgo-search

/bin/bash: line 1: uv: command not found


In [2]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.tools import tool
from dotenv import load_dotenv
import requests

from langchain_community.tools import DuckDuckGoSearchRun
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()


# hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke("Top news in India today")

results

'Home News Today ’s Trending News : Top Headlines in India and ... Today ’s Trending News : Top Headlines in India and World Updates (August 25, 2025) Top - India - news - today ... info@truescoopnews.com ... Copyright © 2025 TrueScoop News . ... News 24 is your stop for those who are inquisitive enough to keep abreast with the ... Today News Headlines in India : Stay Updated with DLSNews24 ... News 24 is your stop for those who are inquisitive enough to keep abreast with the ... Today News Headlines in India : Stay Updated with DLSNews24 Explore the top business news today in India to stay informed about market developments. ... today for in -depth reporting and breaking news from India ...'

In [4]:
@tool 
def get_weather_data(city: str) -> str:
    """
    This function fetches the current weather data for a given city
    """
    url = f"http://api.weatherstack.com/current/access_key=f53d39776b361c65c6c35d5bed0480e4/query={city}"
    

    response = requests.get(url)

    return response.json()


In [5]:
# llm = HuggingFaceEndpoint(
#     repo_id="openai/gpt-oss-20b",
#     # task="text-generation"
#     task="conversational" 
# )

# llm = ChatHuggingFace(llm=llm)

In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",              # or "gemini-1.5-pro" if you have access
    temperature=0,
    convert_system_message_to_human=True
)
# llm = HuggingFaceEndpoint(repo_id="HuggingFaceH4/zephyr-7b-beta")

# chat_model = ChatHuggingFace(llm=llm)

In [7]:
llm.invoke("HI")

/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--a579bb6f-b8d0-419f-9bf9-db1f88a4bc77-0', usage_metadata={'input_tokens': 2, 'output_tokens': 10, 'total_tokens': 162, 'input_token_details': {'cache_read': 0}})

In [8]:
# Step 1: imports
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

In [9]:
# Step 2: Pull the ReAct prompt from Langchain Hub
prompt = hub.pull("hwchase17/react")  # pulls the standard ReAct agent prompt

# prompt.template += """

# IMPORTANT:
# - You MUST strictly follow this format:
# Thought: <your reasoning>
# Action: <tool name, exactly from the list>
# Action Input: <the input for the tool>
# Observation: <the tool result>
# Final Answer: <your final answer>

# - Do NOT put tool names inside Action Input.
# - Do NOT add extra text or explanations outside the format.
# - Only use tools provided in the list.
# """


/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [10]:
# Step 3: Create the ReAct agent manually with the pulled prompt

from langchain.output_parsers import OutputFixingParser
from langchain.agents.output_parsers import ReActSingleInputOutputParser

parser = OutputFixingParser.from_llm(
    parser=ReActSingleInputOutputParser(),
    llm=llm
)

agent = create_react_agent(
    llm=llm, 
    tools=[search_tool, get_weather_data],
    prompt=prompt, 
    output_parser=parser
)

In [11]:
# Step 4: Wrap it with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True,  # see the what agent thinks
    handle_parsing_errors=True
)

In [12]:
# Step 5: Invoke
from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="3 Ways to reach Goa from Delhi")])
print(response)


/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


content="Here are 3 common and practical ways to reach Goa from Delhi:\n\n1.  **By Air (Flight):**\n    *   **How it works:** This is the fastest and most convenient way. You'll fly from Indira Gandhi International Airport (DEL) in Delhi to one of Goa's two airports:\n        *   **Dabolim Airport (GOI):** The older, primary airport, located in South Goa.\n        *   **Manohar International Airport (MOPA):** The newer airport, located in North Goa.\n    *   **Duration:** The direct flight time is approximately **2.5 to 3 hours**. Add 2-3 hours for airport check-in, security, and travel to/from the airports.\n    *   **Airlines:** Major Indian carriers like IndiGo, Vistara, Air India, SpiceJet, and Akasa Air operate frequent direct flights.\n    *   **Pros:** Quickest, most comfortable, saves a lot of travel time.\n    *   **Cons:** Generally the most expensive option, especially if not booked in advance.\n    *   **Tip:** Book your tickets well in advance, especially during peak seaso

In [13]:
# Step 5: Invoke
# response = agent_executor.invoke({"input": "3 Ways to reach goa from Delhi"})
# response = agent_executor.invoke({"input": "Where is todays India and Pakistan Match and also wether of that place"})
response = agent_executor.invoke({"input": "What is the capital of India and its Current GDP and Foreign Reserve and its Population"})
print(response)



> Entering new AgentExecutor chain...


/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


Action: duckduckgo_search
Action Input: capital of IndiaThe national capital of India, New Delhi is jointly administered by both the Central Government of India and the local Government of Delhi, it is also the capital of the National Capital Territory (NCT) of Delhi. Aug 9, 2018 · Learn about the history, geography, and problems of New Delhi, the capital city of India. Find out why and how the capital was moved from Kolkata to New Delhi in 1911. The capital of India is New Delhi , located in the north-central part of the country to the west of the Yamuna River. Calcutta (now Kolkata, the capital of West Bengal) was the country’s capital until 1911when King George V declared Delhi the new capital and construction of New Delhi began. When the national government achieved independence in 1947,... See full list on usatoday.com India is home to 28 states, each with its own capital and run by a Governor who represents the President: 1. Andhra Pradesh 2. Arunachal Pradesh 3. Assam 4. Bihar 5

/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 10
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs

Action: duckduckgo_search
Action Input: India current GDP3 minutes ago — Economy of India ; Increase $4.19 trillion (nominal; 2025 est.) · Increase $17.65 trillion (PPP; 2025 est.) · 4th (nominal; 2025) · 3rd (PPP; 2025) · Increase 6.5% ... 16 Jun 2025 — At current prices, GDP has increased from ₹106.57 lakh crore in 2014–15 to an estimated ₹331.03 lakh crore in 2024–25 , an approximate threefold rise in just ten ... 6 Jul 2025 — With 6.5% GDP growth, India stands as the fastest growing major economy · India's GDP grew 6.5% in 2024–25 , the highest among major economies. · Inflation fell to ... 26 Mar 2025 — According to the IMF, India's GDP currently stands at $4.3 trillion . It was $2.1 trillion in 2015, when Narendra Modi's first term as prime minister was less ... 26 Mar 2025 — India's GDP at current prices stood at USD 2.1 trillion in 2015 and is projected to reach USD 4.27 trillion by the end of 2025, marking a 100 per cent increase ...

/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 10
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 10
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 42
}
]

In [ ]:
print(response["output"])

There is no India vs Pakistan match scheduled for today based on the search results. The information found pertains to an Asia Cup match in 2025 at the Dubai International Stadium.
